# Kaggle GPU End-to-End Runner
This notebook runs the modern-backbone experiment matrix end to end on Kaggle GPU using the packaged local datasets and copied training code.


## Environment Setup
Install dependencies, verify CUDA, and wire notebook paths.


In [ ]:
import os, sys, json, subprocess
from pathlib import Path

ROOT = Path.cwd()
PACK_DIR = ROOT if (ROOT / 'code').exists() else ROOT / 'kaggle_gpu_pack'
CODE_DIR = PACK_DIR / 'code'
DATA_DIR = PACK_DIR / 'datasets' / 'raw'
RESULTS_DIR = CODE_DIR / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print('PACK_DIR:', PACK_DIR)
print('CODE_DIR:', CODE_DIR)
print('DATA_DIR:', DATA_DIR)


In [ ]:
# Kaggle usually already has torch; install missing packages quietly
%pip -q install -U transformers datasets accelerate sentencepiece scikit-learn pandas


In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


## Dataset Staging
Copy packaged datasets into `code/data/raw` where the scripts expect them.


In [ ]:
import shutil
TARGET_RAW = CODE_DIR / 'data' / 'raw'
TARGET_RAW.mkdir(parents=True, exist_ok=True)

for ds in ['atis_iob', 'banking77', 'snips', 'massive']:
    src = DATA_DIR / ds
    dst = TARGET_RAW / ds
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)

if (DATA_DIR / 'clinc150').exists():
    src = DATA_DIR / 'clinc150'
    dst = TARGET_RAW / 'clinc150'
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)

print('Staged datasets:', sorted([p.name for p in TARGET_RAW.iterdir()]))


## Model Configuration
Set model paths. Use Kaggle Dataset paths if you uploaded model folders; otherwise set Hugging Face model IDs.


In [ ]:
MODEL_PATHS = {
    'nemotron': '/kaggle/input/nemotron-mini-4b',  # or 'nvidia/Nemotron-Mini-4B-Instruct'
    'phi': '/kaggle/input/phi-3.5-mini',           # or 'microsoft/Phi-3.5-mini-instruct'
}

if not Path(MODEL_PATHS['nemotron']).exists():
    MODEL_PATHS['nemotron'] = 'nvidia/Nemotron-Mini-4B-Instruct'
if not Path(MODEL_PATHS['phi']).exists():
    MODEL_PATHS['phi'] = 'microsoft/Phi-3.5-mini-instruct'

MODEL_PATHS


## Run Training Matrix
Runs probe sweep + pruned d3 + pruned d12 across datasets and both models. Uses resumable `run_if_missing` semantics.


In [ ]:
import shlex

os.chdir(CODE_DIR)

DATASETS = ['snips', 'massive', 'clinc150', 'banking77']
MODEL_ORDER = [('nemotron', MODEL_PATHS['nemotron']), ('phi', MODEL_PATHS['phi'])]

PROBE_MAX_TRAIN = 120
PROBE_EPOCHS = 10
PRUNED_MAX_TRAIN = 150
PRUNED_EPOCHS = 1
BATCH_SIZE = 2

def run(cmd):
    print('
[RUN]', cmd)
    subprocess.run(cmd, shell=True, check=True)

def run_if_missing(output_path, cmd):
    p = Path(output_path)
    if p.exists():
        print('[SKIP]', output_path)
    else:
        run(cmd)

for model_tag, model_path in MODEL_ORDER:
    for ds in DATASETS:
        out_probe = f'results/{ds}_probe_sweep_usmodern_{model_tag}.json'
        out_d3 = f'results/{ds}_pruned_depth3_usmodern_{model_tag}.json'
        out_d12 = f'results/{ds}_pruned_depth12_usmodern_{model_tag}.json'

        probe_cmd = (
            f'PYTHONPATH=src python scripts/probe_sweep.py --dataset {ds} '
            f'--model-path {shlex.quote(model_path)} --max-train {PROBE_MAX_TRAIN} '
            f'--probe-epochs {PROBE_EPOCHS} --output {out_probe}'
        )
        d3_cmd = (
            f'PYTHONPATH=src python scripts/train_pruned.py --dataset {ds} --depth 3 '
            f'--model-path {shlex.quote(model_path)} --epochs {PRUNED_EPOCHS} --batch-size {BATCH_SIZE} '
            f'--max-train {PRUNED_MAX_TRAIN} --output {out_d3}'
        )
        d12_cmd = (
            f'PYTHONPATH=src python scripts/train_pruned.py --dataset {ds} --depth 12 '
            f'--model-path {shlex.quote(model_path)} --epochs {PRUNED_EPOCHS} --batch-size {BATCH_SIZE} '
            f'--max-train {PRUNED_MAX_TRAIN} --output {out_d12}'
        )

        run_if_missing(out_probe, probe_cmd)
        run_if_missing(out_d3, d3_cmd)
        run_if_missing(out_d12, d12_cmd)

print('Matrix run complete.')


## Results Summary
Load all generated JSON outputs and build a compact summary table.


In [ ]:
import pandas as pd

rows = []
for p in sorted((CODE_DIR / 'results').glob('*_usmodern_*.json')):
    j = json.loads(p.read_text())
    rows.append({
        'file': p.name,
        'dataset': j.get('dataset'),
        'kind': j.get('kind'),
        'depth': j.get('depth'),
        'model': j.get('model'),
        'intent_accuracy': j.get('intent_accuracy'),
        'slot_f1': (j.get('slot_f1') or {}).get('f1') if isinstance(j.get('slot_f1'), dict) else None,
        'train_time_s': j.get('train_time_s'),
        'recommended_depth': j.get('recommended_depth'),
    })

df = pd.DataFrame(rows).sort_values(['dataset', 'model', 'kind', 'depth'], na_position='last')
df.head(50)


In [ ]:
summary_path = PACK_DIR / 'kaggle_run_summary.csv'
df.to_csv(summary_path, index=False)
print('Saved:', summary_path)
print('Artifacts:', len(list((CODE_DIR / 'results').glob('*_usmodern_*.json'))))
